# inbox

> what is waiting for a person, and how far each thing reading on their behalf has got

In [ ]:
#| default_exp inbox

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

This is how an answer reaches you. It also decides the shape of everything else.

In the Agent Client Protocol the editor asks first and the agent answers. The protocol has no call
for an agent to start a turn. So nothing outside PyCharm can make a 5pm roundup appear in an open
PyCharm conversation. A scheduled fire runs on its own instead, and leaves a note. Each place you
work picks that note up on its next turn.

There is more than one such place, which is why reading is per reader. `seq` is a rowid alias, so
a reader's position is one integer. Leela, the terminal and an editor each see every note once.
None of them consumes it for the others.

In [ ]:
#| export
from fastcore.all import AttrDict, L, first, ifnone, patch, xdumps
from pobblebonk.core import BUSY_TIMEOUT_MS, Store, clip, now, row

In [ ]:
#| export
KINDS = ('fire', 'blocked', 'dead', 'note')   # why a note exists; `fire` is an answer, the rest are asks

In [ ]:
#| export
class Reads(Store):
    "Where each reader got to."
    name = 'pob_reads'
    cols = dict(reader=str, seq=int, at=float)
    pk = 'reader'

class Inbox(Store):
    "Notes waiting for a person, each reader keeping its own place."
    name = 'pob_notes'
    cols = dict(seq=int, kind=str, title=str, body=str, ref=str, meta=str, at=float)
    pk = 'seq'    # a rowid alias, so it is monotonic in this file and a cursor is one integer
    idx = ('CREATE INDEX IF NOT EXISTS pob_notes_kind ON pob_notes(kind, seq)',)

    def __init__(self, db, busy_ms:int=BUSY_TIMEOUT_MS):
        super().__init__(db, busy_ms)
        self.reads = Reads(db, busy_ms)

    def __repr__(self): return f'Inbox({self.count()} notes)'

In [ ]:
#| export
@patch
def note(self:Inbox,
         kind:str,          # one of `KINDS`
         title:str,         # one line: which schedule, or what is missing
         body:str='',       # the answer itself, or what is needed
         ref:str=None,      # the schedule or job this came from
         meta:dict=None,
         at:float=None
) -> AttrDict:
    'Leave one note. Nothing is sent anywhere: the next turn picks it up.'
    at = ifnone(at, now())
    with self.txn():
        got = self.db.q('INSERT INTO pob_notes (kind, title, body, ref, meta, at) '
                        'VALUES (?,?,?,?,?,?) RETURNING *',
                        [str(kind), clip(title, 300), clip(body), ref, xdumps(meta), at])
    return row(first(got))

@patch
def seen(self:Inbox, reader:str) -> int:
    'The last `seq` this reader was given, or 0.'
    r = self.reads.one('SELECT seq FROM pob_reads WHERE reader=?', [str(reader)])
    return int(r['seq']) if r else 0

@patch
def unread(self:Inbox, reader:str, limit:int=50) -> L:
    'What this reader has not been given yet, oldest first. Reading does not advance the cursor.'
    return self.q('SELECT * FROM pob_notes WHERE seq>? ORDER BY seq LIMIT ?',
                  [self.seen(reader), limit])

@patch
def mark_read(self:Inbox, reader:str, seq:int=None, at:float=None) -> int:
    'Move this reader to `seq`, or to the newest note. Never backwards.'
    at = ifnone(at, now())
    with self.txn():
        top = self.one('SELECT max(seq) s FROM pob_notes')['s'] or 0
        to = max(self.seen(reader), min(ifnone(seq, top), top))
        self.db.q('INSERT INTO pob_reads (reader, seq, at) VALUES (?,?,?) '
                  'ON CONFLICT(reader) DO UPDATE SET seq=excluded.seq, at=excluded.at',
                  [str(reader), to, at])
    return to

@patch
def drain(self:Inbox, reader:str, limit:int=50, at:float=None) -> L:
    'Every note this reader has not had, taken off its cursor in one go. This is what a turn calls.'
    with self.txn():
        got = self.unread(reader, limit)
        if got: self.mark_read(reader, got[-1]['seq'], at=at)
    return got

@patch
def count(self:Inbox, kind:str=None) -> int:
    'How many notes there are, of one kind or all.'
    w, a = ('WHERE kind=? ', [kind]) if kind else ('', [])
    return self.one(f'SELECT count(*) n FROM pob_notes {w}', a)['n']

@patch
def notes(self:Inbox, kind:str=None, ref:str=None, limit:int=50) -> L:
    'Notes newest first, whatever any reader has seen.'
    w, a = [], []
    if kind: w.append('kind=?'); a.append(kind)
    if ref:  w.append('ref=?');  a.append(ref)
    sql = 'SELECT * FROM pob_notes' + (' WHERE ' + ' AND '.join(w) if w else '')
    return self.q(sql + ' ORDER BY seq DESC LIMIT ?', a + [limit])

@patch
def purge(self:Inbox, before:float) -> int:
    'Delete notes older than `before`. A reader cursor past the end is harmless.'
    with self.txn():
        return len(self.db.q('DELETE FROM pob_notes WHERE at<? RETURNING seq', [before]))

Two fires land while nobody is looking:

In [ ]:
from pobblebonk.core import vault

ib = Inbox(vault())
ib.note('fire', 'git roundup', '9 commits, down from 14 last week', ref='sch1')
ib.note('blocked', 'news needs your interests', 'Add to the interests list and it runs next time.')
test_eq(ib.count(), 2)
test_eq(ib.seen('leela'), 0)
test_eq(ib.unread('leela').attrgot('title'), ['git roundup', 'news needs your interests'])

Each reader gets everything once, and one reading does not consume it for the others:

In [ ]:
test_eq(ib.drain('leela').attrgot('kind'), ['fire', 'blocked'])
test_eq(ib.drain('leela'), [])                       # leela is up to date
test_eq(len(ib.drain('terminal')), 2)                # the terminal has still seen nothing
test_eq(len(ib.drain('pycharm')), 2)

A note left after that reaches all three:

In [ ]:
ib.note('fire', 'cart filled', 'added 6 items, no oat milk')
for r in ('leela', 'terminal', 'pycharm'): test_eq(ib.drain(r).attrgot('title'), ['cart filled'])

A cursor only ever moves forward, so a reader cannot be made to see a note twice by asking:

In [ ]:
test_eq(ib.mark_read('leela', 1), 3)     # asked for 1, stays at 3
test_eq(ib.drain('leela'), [])
test_eq(ib.mark_read('leela', 999), 3)   # and cannot run past the newest note

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()